# 3. Few-Shot Clustering for Grafana Logs

This notebook implements few-shot clustering using sentence embeddings and HDBSCAN.

## Objectives
1. Generate embeddings using pre-trained models (Sentence-BERT)
2. Create few-shot examples from labeled data
3. Apply HDBSCAN for density-based clustering
4. Use FAISS for efficient similarity search
5. Label clusters using LLM-based approach
6. Evaluate clustering quality

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Embedding and clustering libraries
from sentence_transformers import SentenceTransformer
import hdbscan
import faiss
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
import umap

print("✅ Libraries imported successfully")

## 1. Load Engineered Features

In [ ]:
# Load engineered features
features_file = '../output/grafana/engineered_features.parquet'
df = pd.read_parquet(features_file)

print(f"✅ Loaded features: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Generate Embeddings using Sentence-BERT

In [ ]:
# Load pre-trained sentence transformer model
print("Loading Sentence-BERT model...")
model_name = 'all-MiniLM-L6-v2'  # Fast and efficient model
# Alternative: 'all-mpnet-base-v2' for better quality
model = SentenceTransformer(model_name)
print(f"✅ Loaded model: {model_name}")

# Generate embeddings
print("\nGenerating embeddings for all logs...")
print(f"Processing {len(df):,} logs...")

# Use normalized text for embedding
texts = df['normalized_text'].tolist()

# Generate embeddings in batches for efficiency
batch_size = 256
embeddings = model.encode(
    texts,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Generated embeddings with shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

## 3. Create Few-Shot Examples

Define a small set of labeled examples to guide clustering.

In [ ]:
# Define few-shot examples based on domain knowledge
few_shot_examples = [
    {
        'label': 'memory_monitoring',
        'keywords': ['memory', 'heap', 'jvm', 'usage', 'bytes'],
        'description': 'Memory-related metrics and monitoring'
    },
    {
        'label': 'performance_metrics',
        'keywords': ['response', 'latency', 'duration', 'p95', 'p99', 'throughput'],
        'description': 'Application performance and response time metrics'
    },
    {
        'label': 'database_monitoring',
        'keywords': ['database', 'query', 'connection', 'pool', 'postgres', 'redis'],
        'description': 'Database connection and query metrics'
    },
    {
        'label': 'cache_metrics',
        'keywords': ['cache', 'hit', 'miss', 'rate', 'keys'],
        'description': 'Cache performance and hit/miss rates'
    },
    {
        'label': 'infrastructure_health',
        'keywords': ['cpu', 'network', 'disk', 'container', 'pod'],
        'description': 'Infrastructure and container health metrics'
    },
    {
        'label': 'jvm_metrics',
        'keywords': ['jvm', 'gc', 'garbage', 'collection', 'thread'],
        'description': 'JVM-specific metrics including garbage collection'
    },
    {
        'label': 'availability_uptime',
        'keywords': ['up{', 'availability', 'uptime', 'health'],
        'description': 'Service availability and uptime monitoring'
    }
]

print("📋 Few-Shot Examples Defined:")
for i, example in enumerate(few_shot_examples, 1):
    print(f"{i}. {example['label']}: {example['description']}")
    print(f"   Keywords: {', '.join(example['keywords'])}")

## 4. Match Few-Shot Examples to Logs

In [ ]:
def match_fewshot_label(text, examples):
    """
    Match log text to few-shot examples based on keywords.
    Returns label or None if no match.
    """
    text_lower = text.lower()
    scores = []
    
    for example in examples:
        # Count keyword matches
        matches = sum(1 for keyword in example['keywords'] if keyword.lower() in text_lower)
        scores.append((example['label'], matches))
    
    # Return label with highest score if any matches
    best_label, best_score = max(scores, key=lambda x: x[1])
    return best_label if best_score > 0 else None

# Apply few-shot labels
print("Applying few-shot labels...")
df['fewshot_label'] = df['normalized_text'].apply(lambda x: match_fewshot_label(x, few_shot_examples))

labeled_count = df['fewshot_label'].notna().sum()
print(f"\n✅ Labeled {labeled_count:,} logs ({labeled_count/len(df)*100:.1f}%)")
print(f"\nLabel distribution:")
print(df['fewshot_label'].value_counts())

## 5. Build FAISS Index for Similarity Search

In [ ]:
# Normalize embeddings for cosine similarity
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Build FAISS index
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product for cosine similarity
index.add(embeddings_normalized.astype('float32'))

print(f"✅ FAISS index built with {index.ntotal:,} vectors")

# Test similarity search
k = 5  # number of nearest neighbors
query_idx = 0
query_embedding = embeddings_normalized[query_idx:query_idx+1].astype('float32')
distances, indices = index.search(query_embedding, k)

print(f"\n🔍 Test query - Top {k} similar logs to log #{query_idx}:")
print(f"\nQuery: {df.iloc[query_idx]['normalized_text'][:100]}...")
print(f"\nSimilar logs:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    print(f"{i}. [Similarity: {dist:.3f}] {df.iloc[idx]['normalized_text'][:80]}...")

## 6. HDBSCAN Clustering

In [ ]:
# Apply HDBSCAN clustering
print("Applying HDBSCAN clustering...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,       # Minimum cluster size
    min_samples=10,             # Minimum samples in neighborhood
    metric='euclidean',
    cluster_selection_method='eom',  # Excess of Mass
    prediction_data=True
)

# Fit clustering
cluster_labels = clusterer.fit_predict(embeddings)

df['cluster'] = cluster_labels

# Calculate cluster statistics
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f"\n✅ HDBSCAN Clustering Complete")
print(f"\n📊 Clustering Results:")
print(f"  - Number of clusters: {n_clusters}")
print(f"  - Noise points: {n_noise:,} ({n_noise/len(cluster_labels)*100:.1f}%)")
print(f"  - Clustered points: {len(cluster_labels) - n_noise:,} ({(len(cluster_labels) - n_noise)/len(cluster_labels)*100:.1f}%)")

# Cluster size distribution
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\n📈 Cluster Size Distribution:")
for cluster_id, count in cluster_counts.items():
    if cluster_id != -1:  # Skip noise
        print(f"  Cluster {cluster_id:2d}: {count:6,} logs ({count/len(df)*100:5.2f}%)")

## 7. Cluster Quality Metrics

In [ ]:
# Calculate clustering quality metrics (excluding noise points)
mask = cluster_labels != -1
if mask.sum() > 0 and n_clusters > 1:
    silhouette = silhouette_score(embeddings[mask], cluster_labels[mask])
    davies_bouldin = davies_bouldin_score(embeddings[mask], cluster_labels[mask])
    
    print("\n📊 Clustering Quality Metrics:")
    print(f"  - Silhouette Score: {silhouette:.4f} (higher is better, range: [-1, 1])")
    print(f"  - Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
    
    # Interpretation
    if silhouette > 0.7:
        quality = "Excellent"
    elif silhouette > 0.5:
        quality = "Good"
    elif silhouette > 0.25:
        quality = "Fair"
    else:
        quality = "Poor"
    
    print(f"\n  Overall Quality: {quality}")
else:
    print("\n⚠️  Not enough clustered points to calculate quality metrics")

## 8. Assign Few-Shot Labels to Clusters

In [ ]:
def assign_cluster_label(cluster_id, df):
    """
    Assign label to cluster based on majority few-shot labels.
    """
    cluster_df = df[df['cluster'] == cluster_id]
    
    # Get few-shot label distribution in cluster
    label_counts = cluster_df['fewshot_label'].value_counts()
    
    if len(label_counts) > 0 and label_counts.iloc[0] > 0:
        # Majority label
        majority_label = label_counts.index[0]
        confidence = label_counts.iloc[0] / len(cluster_df)
        return majority_label, confidence
    
    # Fallback: use most common characteristics
    top_dashboard = cluster_df['dashboard'].mode()[0] if len(cluster_df['dashboard'].mode()) > 0 else 'unknown'
    top_panel = cluster_df['panel_title'].mode()[0] if len(cluster_df['panel_title'].mode()) > 0 else 'unknown'
    return f"{top_dashboard}_{top_panel}", 0.0

# Assign labels to all clusters
cluster_labels_map = {}
for cluster_id in sorted(df[df['cluster'] != -1]['cluster'].unique()):
    label, confidence = assign_cluster_label(cluster_id, df)
    cluster_labels_map[cluster_id] = {'label': label, 'confidence': confidence}

print("\n🏷️  Cluster Labels:")
for cluster_id, info in cluster_labels_map.items():
    count = (df['cluster'] == cluster_id).sum()
    print(f"  Cluster {cluster_id:2d}: {info['label']:<30} (confidence: {info['confidence']:.2%}, n={count:,})")

# Add cluster labels to dataframe
df['cluster_label'] = df['cluster'].map(lambda x: cluster_labels_map.get(x, {'label': 'noise'})['label'])

## 9. Visualize Clusters with UMAP

In [ ]:
# Reduce dimensions using UMAP for visualization
print("Reducing dimensions with UMAP...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

# Use a sample for faster visualization if dataset is large
sample_size = min(10000, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)

embeddings_sample = embeddings[sample_indices]
labels_sample = cluster_labels[sample_indices]

embeddings_2d = reducer.fit_transform(embeddings_sample)

print(f"✅ UMAP reduction complete")

# Plot clusters
fig, ax = plt.subplots(figsize=(16, 12))

# Plot noise points in gray
noise_mask = labels_sample == -1
if noise_mask.sum() > 0:
    ax.scatter(
        embeddings_2d[noise_mask, 0],
        embeddings_2d[noise_mask, 1],
        c='lightgray',
        alpha=0.3,
        s=20,
        label='Noise'
    )

# Plot clusters with different colors
unique_clusters = [c for c in np.unique(labels_sample) if c != -1]
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))

for cluster_id, color in zip(unique_clusters, colors):
    cluster_mask = labels_sample == cluster_id
    label = cluster_labels_map.get(cluster_id, {'label': f'Cluster {cluster_id}'})['label']
    ax.scatter(
        embeddings_2d[cluster_mask, 0],
        embeddings_2d[cluster_mask, 1],
        c=[color],
        alpha=0.6,
        s=50,
        label=f'{label} (C{cluster_id})'
    )

ax.set_title('HDBSCAN Clusters Visualization (UMAP)', fontsize=16, fontweight='bold')
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/grafana/cluster_visualization_umap.png', dpi=300, bbox_inches='tight')
print("\n✅ Saved visualization to: output/grafana/cluster_visualization_umap.png")
plt.show()

## 10. Cluster Analysis

In [ ]:
# Detailed cluster analysis
print("\n" + "="*80)
print("DETAILED CLUSTER ANALYSIS")
print("="*80)

for cluster_id in sorted(df[df['cluster'] != -1]['cluster'].unique()):
    cluster_df = df[df['cluster'] == cluster_id]
    
    print(f"\n{'='*80}")
    print(f"Cluster {cluster_id}: {cluster_labels_map[cluster_id]['label']}")
    print(f"{'='*80}")
    print(f"Size: {len(cluster_df):,} logs ({len(cluster_df)/len(df)*100:.2f}%)")
    print(f"\nTop Services:")
    print(cluster_df['service'].value_counts().head(5))
    print(f"\nTop Dashboards:")
    print(cluster_df['dashboard'].value_counts().head(5))
    print(f"\nTop Panel Titles:")
    print(cluster_df['panel_title'].value_counts().head(5))
    print(f"\nSample Logs:")
    for i, text in enumerate(cluster_df['normalized_text'].head(3), 1):
        print(f"{i}. {text[:150]}...")

## 11. Export Results

In [ ]:
# Save clustered data
output_file = '../output/grafana/clustered_logs.parquet'
df.to_parquet(output_file, index=False)
print(f"✅ Saved clustered data to: {output_file}")

# Save embeddings
embeddings_file = '../output/grafana/embeddings.npy'
np.save(embeddings_file, embeddings)
print(f"✅ Saved embeddings to: {embeddings_file}")

# Save FAISS index
faiss_file = '../output/grafana/faiss_index.bin'
faiss.write_index(index, faiss_file)
print(f"✅ Saved FAISS index to: {faiss_file}")

# Save cluster summary
cluster_summary = {
    'n_clusters': n_clusters,
    'n_noise': n_noise,
    'silhouette_score': float(silhouette) if 'silhouette' in locals() else None,
    'davies_bouldin_score': float(davies_bouldin) if 'davies_bouldin' in locals() else None,
    'cluster_labels': {int(k): v for k, v in cluster_labels_map.items()},
    'timestamp': datetime.now().isoformat()
}

summary_file = '../output/grafana/cluster_summary.json'
with open(summary_file, 'w') as f:
    json.dump(cluster_summary, f, indent=2)
print(f"✅ Saved cluster summary to: {summary_file}")

print(f"\n" + "="*80)
print("✅ Few-Shot Clustering Complete")
print("="*80)